# Analyse Exploratoire et Modélisation NLP + Régression

# Étape 1 : Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import kagglehub
import pathlib
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow_hub as hub
import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertModel

# Étape 2 : Chargement des données


# Chargement du fichier CSV principal


In [ ]:
path = kagglehub.dataset_download("PromptCloudHQ/imdb-data")
path = pathlib.Path(path)

# Etape 3: Charger Le CSV


In [ ]:
print("Fichiers disponibles dans le dataset:")
for file in path.iterdir():
    print("-", file.name)


# Étape 4 : Nettoyage de texte


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    return text

df['clean_overview'] = df['Overview'].apply(clean_text)

In [ ]:
csv_file = path / "IMDB-Movie-Data.csv"
df = pd.read_csv(csv_file)
df = df[['Title', 'Description', 'Rating']]
df.dropna(inplace=True)
df = df[df['Description'].str.len() > 10]
df = df.rename(columns={
    'Title': 'Series_Title',
    'Description': 'Overview',
    'Rating': 'IMDB_Rating'
})
df['clean_overview'] = df['Overview'].apply(clean_text)

# Étape 5 : Analyse exploratoire


In [ ]:
print(df.describe())
df['IMDB_Rating'].hist(bins=20)
plt.title("Distribution des notes IMDb")
plt.xlabel("Note")
plt.ylabel("Nombre de films")
plt.show()

# Étape 6 : Modèle TF-IDF + Ridge Regression


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_overview'], df['IMDB_Rating'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

ridge = Ridge()
ridge.fit(X_train_tfidf, y_train)
preds_ridge = ridge.predict(X_test_tfidf)

print("\n--- Ridge TF-IDF ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds_ridge)))
print("MAE:", mean_absolute_error(y_test, preds_ridge))
print("R2:", r2_score(y_test, preds_ridge))

# Étape 7 : Universal Sentence Encoder + DNN


In [ ]:
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
X_train_embed = embed(X_train.tolist())
X_test_embed = embed(X_test.tolist())

model_use = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(512,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_use.compile(optimizer='adam', loss='mse')
model_use.fit(X_train_embed, y_train, epochs=5, batch_size=32)
preds_use = model_use.predict(X_test_embed).flatten()

print("\n--- Universal Sentence Encoder ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds_use)))
print("MAE:", mean_absolute_error(y_test, preds_use))
print("R2:", r2_score(y_test, preds_use))

# Étape 8: Modélisation avec DistilBERT


In [ ]:
# --- Étape 8 : Modélisation avec DistilBERT (régression) ---
!pip install transformers torch

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Sous-échantillon rapide (par exemple 5000 échantillons)
df_small = df.sample(min(1000, len(df)), random_state=42).reset_index(drop=True)
# 2. Dataset PyTorch
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class IMDbDataset(Dataset):
    def __init__(self, texts, targets):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.targets = targets.values.astype(np.float32)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        return item, torch.tensor(self.targets[idx])

# 3. Split train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df_small['clean_overview'], df_small['IMDB_Rating'], test_size=0.2, random_state=42)

train_ds = IMDbDataset(X_train, y_train)
test_ds  = IMDbDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=16)

# 4. Modèle : DistilBERT + couche de régression
class DistilBertRegressor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = torch.nn.Dropout(0.3)
        self.out = torch.nn.Linear(self.bert.config.hidden_size, 1)
    def forward(self, input_ids, attention_mask):
        hidden = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0][:,0]
        return self.out(self.dropout(hidden)).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertRegressor().to(device)
optim = torch.optim.Adam(model.parameters(), lr=2e-5)
loss_fn = torch.nn.MSELoss()

# 5. Boucle d'entraînement (3 époques)
for epoch in range(3):
    model.train()
    epoch_losses = []
    for batch, target in train_loader:
        optim.zero_grad()
        preds = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = loss_fn(preds, target.to(device))
        loss.backward()
        optim.step()
        epoch_losses.append(loss.item())
    print(f"Époch {epoch+1} — loss moyenne : {np.mean(epoch_losses):.4f}")

# 6. Évaluation
model.eval()
y_pred = []
y_true = []
with torch.no_grad():
    for batch, target in test_loader:
        preds = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(target.numpy())

mae_distil = mean_absolute_error(y_true, y_pred)
rmse_distil = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"DistilBERT MAE: {mae_distil:.3f}, RMSE: {rmse_distil:.3f}")


In [ ]:
!pip install -q --upgrade streamlit pyngrok

In [ ]:
!pip install --upgrade jinja2 streamlit altair pyngrok --quiet

In [ ]:
import jinja2, flask, streamlit, altair, sys
print("jinja2 :", jinja2.__version__)
print("flask  :", flask.__version__)
print("streamlit :", streamlit.__version__)
print("altair :", altair.__version__)
print("python :", sys.version)

In [ ]:
!pip install "flask<2.3" "sphinx<7.2" --quiet

In [ ]:

import jinja2, flask, streamlit, altair, sys
print("jinja2    :", jinja2.__version__)
print("flask     :", flask.__version__)
print("streamlit :", streamlit.__version__)
print("altair    :", altair.__version__)
print("python    :", sys.version)

!pip install -q \
  "streamlit==1.33.0" \
  "jinja2==3.0.3" \
  "flask<2.3" \
  "markupsafe==2.1.4" \
  "sphinx<7.2" \
  "pyngrok==6.0.0"

In [ ]:
%%writefile app.py
import streamlit as st
st.set_page_config(page_title="Prédicteur IMDb", layout="centered")
st.title("Prédicteur IMDb : démo")
st.write("Test OK ! – ajoute ton modèle ici")

In [ ]:
%%writefile app.py
"""
Prédicteur IMDb
-------------------------------------------
- Patch Jinja2 → rétablit `Markup`
- Charge DistilBERT une seule fois (→ ~10 s)
"""

#  correctif Jinja2
import jinja2, markupsafe
if not hasattr(jinja2, "Markup"):          # Jinja2 ≥ 3.1 ?
    jinja2.Markup = markupsafe.Markup      # on le remet !

# reste de l'app Streamlit
import streamlit as st
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf

st.set_page_config(page_title="Prédicteur IMDb", layout="centered")
st.title("Prédicteur IMDb : démo ⏭️ sans ngrok")

@st.cache_resource(show_spinner="⏳ Téléchargement du modèle…")
def load_model():
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    mdl = TFAutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased-finetuned-sst-2-english", num_labels=1
    )
    return tok, mdl

tokenizer, model = load_model()

txt = st.text_area("Votre critique (en anglais)", height=150)
if st.button("Prédire la note") and txt.strip():
    inputs = tokenizer(txt, return_tensors="tf", truncation=True, padding=True)
    raw    = model(**inputs).logits[0][0].numpy().item()
    score  = max(0, min(10, raw*2 + 5))
    st.metric("Note prédite (0–10)", f"{score:.1f}")
elif not txt.strip():
    st.info("➡️  Saisis d’abord un texte.")

In [ ]:
%%bash --bg
streamlit run app.py \
  --server.headless true \
  --server.port 8501 \
  --server.address localhost

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=800)